# GrowthParameterEstimation Function Tour

This notebook is a broad executable tour of the package: public API inventory, data validation, exposure schedules, composable models, registry models, simulation, sweeps, fitting APIs, statistical analysis, joint fitting, staged workflows, and figure-heavy result output.

Manufactured data progress from logistic and Gompertz monocultures into Hill drug response, cooperative coculture, and cooperative drug-death coculture models.


In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))
using GrowthParameterEstimation
using CSV, DataFrames, DifferentialEquations, Random, Statistics, StatsBase, Distributions, Plots, Dates
const GPE = GrowthParameterEstimation
const OUTDIR = joinpath(@__DIR__, "..", "outputs", "function_tour")
const FIGDIR = joinpath(OUTDIR, "figures")
const TABLEDIR = joinpath(OUTDIR, "tables")
const DIAGDIR = joinpath(OUTDIR, "diagnostics")
for d in (OUTDIR, FIGDIR, TABLEDIR, DIAGDIR); mkpath(d); end
default(size=(1100, 650), linewidth=2.5, markersize=4, legend=:best)
Random.seed!(20260722)
save_table(name, df) = (p = joinpath(TABLEDIR, name); CSV.write(p, df); p)
save_fig(name, plt) = (p = joinpath(FIGDIR, name); savefig(plt, p); p)
status_row(area, cap, ok, out=""; detail="") = (area=String(area), capability=String(cap), ok=Bool(ok), output=String(out), detail=String(detail))
tour_status = NamedTuple[]; produced_paths = String[]; figure_paths = String[]
println("Output directory: ", OUTDIR)


## Public API Inventory


In [ ]:
api_df = DataFrame(symbol=sort(String.(names(GPE; all=false))))
push!(produced_paths, save_table("public_api_symbols.csv", api_df))
mods = [GPE.DataLayer, GPE.Exposure, GPE.Models, GPE.Registry, GPE.Simulation, GPE.Observation, GPE.Fitting, GPE.Analysis, GPE.Workflow]
mod_names = ["DataLayer", "Exposure", "Models", "Registry", "Simulation", "Observation", "Fitting", "Analysis", "Workflow"]
module_exports = DataFrame(module_name=mod_names, symbols=[join(sort(String.(names(m; all=false))), ", ") for m in mods])
push!(produced_paths, save_table("module_exports.csv", module_exports))
display(module_exports)
push!(tour_status, status_row("inventory", "public API and module export inventory", true, "tables/public_api_symbols.csv"))


## Manufactured Multistage Dataset


In [ ]:
times = collect(0.0:1.0:8.0)
function simreg(name, params, u0; dose=0.0)
    spec = get_model(name)
    sim = simulate(spec, times, params; u0=u0, exposure=ConstantExposure(dose), reltol=1e-8, abstol=1e-8)
    sim.success || error("Simulation failed for $name: $(sim.reason)")
    sim
end
function addcond!(rows, sim; culture_type, population_type, dose, treatment_amount, density, replicate, model_generator, true_parameters)
    for (i,t) in enumerate(sim.times)
        latent = sim.observed[i]
        obs = max(latent * (1 + 0.035sin(0.9t + replicate) + 0.015cos(0.41replicate + t)), 1e-6)
        st = sim.states[:, i]
        push!(rows, (time=t, count=obs, error=max(0.06obs, 1.0), dose=dose, cell_line="A549", density=density,
            replicate=replicate, culture_type=culture_type, population_type=population_type, treatment_amount=treatment_amount,
            model_generator=model_generator, true_parameters=true_parameters,
            latent_sensitive=length(st) >= 1 ? st[1] : latent, latent_resistant=length(st) >= 2 ? st[2] : 0.0,
            condition_name=join([culture_type, population_type, string(treatment_amount), density, string(replicate)], "_")))
    end
end
rows = NamedTuple[]
for rep in 1:3
    addcond!(rows, simreg("logistic_growth", [0.55,1200.0], [80.0]); culture_type="monoculture", population_type="naive", dose=0.0, treatment_amount=0.0, density=0.5, replicate=rep, model_generator="logistic_growth", true_parameters="r=0.55;K=1200")
    addcond!(rows, simreg("gompertz_growth", [0.34,0.0,1500.0], [75.0]); culture_type="monoculture", population_type="resistant", dose=0.0, treatment_amount=0.0, density=0.5, replicate=rep, model_generator="gompertz_growth", true_parameters="a=0.34;b=0;K=1500")
    addcond!(rows, simreg("theta_logistic_hill_kill", [0.60,1200.0,1.0,0.16,2.4,1.6], [85.0]; dose=2.5); culture_type="monoculture", population_type="naive", dose=2.5, treatment_amount=2.5, density=0.5, replicate=rep, model_generator="theta_logistic_hill_kill", true_parameters="r=0.60;K=1200;theta=1.0;EmaxKill=0.16;EC50=2.4;hill=1.6")
    addcond!(rows, simreg("theta_logistic_hill_inhibition", [0.38,1500.0,0.65,4.5,1.2], [75.0]; dose=2.5); culture_type="monoculture", population_type="resistant", dose=2.5, treatment_amount=2.5, density=0.5, replicate=rep, model_generator="theta_logistic_hill_inhibition", true_parameters="r=0.38;K=1500;Emax=0.65;EC50=4.5;hill=1.2")
    addcond!(rows, simreg("lotka_volterra_cooperation", [0.44,900.0,0.08,0.31,760.0,0.05], [65.0,45.0]); culture_type="coculture", population_type="mixed", dose=0.0, treatment_amount=0.0, density=1.0, replicate=rep, model_generator="lotka_volterra_cooperation", true_parameters="rS=0.44;KS=900;betaSR=0.08;rR=0.31;KR=760;betaRS=0.05")
    addcond!(rows, simreg("lotka_volterra_hill_cooperation", [0.46,900.0,0.06,0.32,760.0,0.05,0.14,2.1,0.05,6.0,1.5], [65.0,45.0]; dose=2.5); culture_type="coculture", population_type="mixed", dose=2.5, treatment_amount=2.5, density=1.0, replicate=rep, model_generator="lotka_volterra_hill_cooperation", true_parameters="rS=0.46;KS=900;betaSR=0.06;rR=0.32;KR=760;betaRS=0.05;EmaxS=0.14;IC50S=2.1;EmaxR=0.05;IC50R=6.0;hill=1.5")
end
manufactured = DataFrame(rows); sort!(manufactured, [:culture_type,:population_type,:treatment_amount,:replicate,:time])
manufactured_path = joinpath(OUTDIR, "manufactured_multistage_data.csv"); CSV.write(manufactured_path, manufactured); push!(produced_paths, manufactured_path)
story = combine(groupby(manufactured, [:culture_type,:population_type,:treatment_amount,:model_generator,:true_parameters]), nrow=>:n_rows)
push!(produced_paths, save_table("manufactured_model_story.csv", story)); display(story)
push!(tour_status, status_row("manufactured data", "logistic/Gompertz/Hill/cooperation/cooperation-drug generators", true, "manufactured_multistage_data.csv"))


## Data Layer, QC, Preflight, And Exposure


In [ ]:
normalized = normalize_schema(load_timeseries(manufactured_path)); sort!(normalized, :time)
validate_timeseries(normalized); validate_required_metadata(normalized)
dataset_summary = summarize_datasets(normalized); push!(produced_paths, save_table("dataset_summary.csv", dataset_summary))
qc_paths = save_qc_report(generate_qc_report(normalized); output_dir=DIAGDIR); append!(produced_paths, collect(values(qc_paths)))
preflight = preflight_data_quality(normalized; stages=default_stages())
preflight_paths = save_preflight_report(preflight; output_dir=DIAGDIR); append!(produced_paths, collect(values(preflight_paths)))
grid = collect(0.0:0.1:8.0)
exps = Dict("constant"=>ConstantExposure(2.5), "pulse"=>PulseExposure(5.0,1.5,4.5), "stepped"=>SteppedExposure([0.0,2.0,5.0],[0.0,2.5,1.0]), "decaying"=>DecayingExposure(5.0,0.42,0.0), "built_constant"=>build_exposure(:constant; value=1.75))
exposure_df = reduce(vcat, [DataFrame(time=grid, exposure=k, value=evaluate_exposure(v, grid)) for (k,v) in exps])
push!(produced_paths, save_table("exposure_profiles.csv", exposure_df))
p = plot(title="Exposure schedules", xlabel="Time", ylabel="Exposure")
for g in groupby(exposure_df, :exposure); plot!(p, g.time, g.value, label=first(g.exposure)); end
push!(figure_paths, save_fig("exposure_profiles.png", p))
display(preflight.summary)
push!(tour_status, status_row("data and exposure", "load/normalize/validate/summarize/QC/preflight/exposure", preflight.ready_for_fit, "figures/exposure_profiles.png"))


## Composable Models, Observation, Registry, Simulation, And Sweep


In [ ]:
base = GPE.Models.build_logistic(r=0.55, K=1200.0)
models = [("logistic", base), ("gompertz", GPE.Models.build_gompertz(a=0.34,b=0.0,K=1500.0)), ("exponential", GPE.Models.build_exponential(r=0.30)),
    ("death", GPE.Models.apply_death(base; death_rate=0.08)), ("lag", GPE.Models.apply_lag(base; tlag=1.5)),
    ("hill_inhibition", GPE.Models.apply_hill_inhibition(base; emax=0.75, ic50=2.5, hill=1.4)),
    ("hill_kill", GPE.Models.apply_hill_kill(base; emax_kill=0.85, ic50=2.0, hill=1.5)),
    ("composite", GPE.Models.compose_models(base, [GPE.Models.DeathModifier, GPE.Models.LagPhaseModifier]; death_rate=0.05, tlag=1.0))]
u_grid = collect(range(20.0, 1100.0, length=160))
deriv_df = reduce(vcat, [DataFrame(population=u_grid, model=n, derivative=[m(u,Dict(:drug=>2.0),2.0) for u in u_grid]) for (n,m) in models])
push!(produced_paths, save_table("composable_model_derivatives.csv", deriv_df))
pder = plot(title="Composable model derivative shapes", xlabel="Population", ylabel="du/dt")
for g in groupby(deriv_df, :model); plot!(pder, g.population, g.derivative, label=first(g.model)); end
push!(figure_paths, save_fig("composable_model_derivatives.png", pder))
obs_examples = DataFrame(mapping=["viable_total","sum_states_1_2","scaled_background"], value=[viable_total([12.0,4.0],nothing,0.0), sum_states([1,2])([12.0,4.0,9.0],nothing,0.0), observed_signal(ObservationSpec("scaled_total", viable_total, 1.25, 2.0), [12.0,4.0], nothing, 0.0)])
push!(produced_paths, save_table("observation_examples.csv", obs_examples))
reg = DataFrame(model=list_models()); reg.family=[get_model(m).base_growth_family for m in reg.model]; reg.n_states=[get_model(m).n_states for m in reg.model]; reg.n_params=[length(get_model(m).param_names) for m in reg.model]; reg.params=[join(String.(get_model(m).param_names), ";") for m in reg.model]
push!(produced_paths, save_table("registry_inventory.csv", reg)); push!(produced_paths, save_table("registry_families.csv", combine(groupby(reg,:family), nrow=>:n_models)))
mid(spec) = [sqrt(max(lo,1e-9)*max(hi,1e-9)) for (lo,hi) in spec.bounds]
galrows = NamedTuple[]
for name in reg.model
    spec = get_model(name); exposure = occursin("hill", name) || occursin("ic50", name) || occursin("pkpd", name) || occursin("kill", name) ? ConstantExposure(2.5) : ConstantExposure(0.0)
    sim = simulate(spec, times, mid(spec); u0=spec.n_states==1 ? [80.0] : fill(45.0, spec.n_states), exposure=exposure, reltol=1e-6, abstol=1e-6)
    for (i,t) in enumerate(times); push!(galrows, (model=name, family=spec.base_growth_family, time=t, observed=sim.success ? sim.observed[i] : NaN, success=sim.success, reason=sim.reason)); end
end
gallery = DataFrame(galrows); push!(produced_paths, save_table("registered_model_simulation_gallery.csv", gallery))
pgal = plot(title="Registered model simulation gallery", xlabel="Time", ylabel="Observed", legend=false)
for g in groupby(gallery[gallery.success .== true, :], :model); plot!(pgal, g.time, g.observed, alpha=0.55); end
push!(figure_paths, save_fig("registered_model_simulation_gallery.png", pgal))
sweep = run_sweep(get_model("theta_logistic_hill_kill"), [0.55,1200.0,1.0,0.12,2.2,1.5], SweepGrid([60.0,100.0], [0.0], [0.0,1.0,2.5,5.0], times))
sweep_df = sweep.summary; push!(produced_paths, save_table("dose_sweep_summary.csv", sweep_df))
psw = plot(title="Dose sweep: Hill kill model", xlabel="Time", ylabel="Observed")
for r in eachrow(sweep_df)
    key = (seed_total=r.seed_total, resistant_fraction=r.resistant_fraction, dose=r.dose)
    if r.success && haskey(sweep.simulations, key)
        sim = sweep.simulations[key]
        plot!(psw, sim.times, sim.observed, label="dose=$(r.dose), u0=$(r.seed_total)")
    end
end
push!(figure_paths, save_fig("dose_sweep_hill_kill.png", psw))
push!(tour_status, status_row("models/simulation", "builders/modifiers/observation/registry/simulate/run_sweep", true, "figures/registered_model_simulation_gallery.png"))


## Fitting API Tour


In [ ]:
fit_rows = NamedTuple[]
mono_naive = subset(normalized, :culture_type=>ByRow(==("monoculture")), :population_type=>ByRow(==("naive")), :dose=>ByRow(==(0.0)), :replicate=>ByRow(==(1)))
mono_treated = subset(normalized, :culture_type=>ByRow(==("monoculture")), :population_type=>ByRow(==("naive")), :dose=>ByRow(>(0.0)), :replicate=>ByRow(==(1)))
mono_resistant = subset(normalized, :culture_type=>ByRow(==("monoculture")), :population_type=>ByRow(==("resistant")), :dose=>ByRow(==(0.0)), :replicate=>ByRow(==(1)))
x = Float64.(mono_naive.time); y = Float64.(mono_naive.count); tx = Float64.(mono_treated.time); ty = Float64.(mono_treated.count); rx = Float64.(mono_resistant.time); ry = Float64.(mono_resistant.count)
logistic_ode = GPE.Models.to_ode!(GPE.Models.build_logistic())
gompertz_ode = GPE.Models.to_ode!(GPE.Models.build_gompertz())
exponential_ode = GPE.Models.to_ode!(GPE.Models.build_exponential())
low_bounds = [(0.01,1.2),(200.0,2500.0)]
popt, solopt, probopt = setUpProblem(logistic_ode, x, y, Tsit5(), [y[1]], [0.35,1000.0], (x[1],x[end]), low_bounds; maxiters=700)
lbic, lssr = calculate_bic(probopt, x, y, Tsit5(), popt); pQuickStat(x, y, popt, solopt, probopt, lbic, lssr)
push!(produced_paths, save_table("low_level_fit_summary.csv", DataFrame(function_name=["setUpProblem","calculate_bic","pQuickStat"], value=[string(popt), string(lbic), string(lssr)])))
push!(fit_rows, (capability="setUpProblem/calculate_bic/pQuickStat", ok=true, bic=lbic, ssr=lssr, output="tables/low_level_fit_summary.csv"))
single_fit = run_single_fit(x, y, [0.35,1000.0]; model=logistic_ode, bounds=low_bounds, solver=Tsit5(), max_time=20.0, show_stats=false)
push!(fit_rows, (capability="run_single_fit", ok=true, bic=single_fit.bic, ssr=single_fit.ssr, output=""))
reg_fit = fit_model(get_model("logistic_growth"), x, y, 0.0; p0=[0.45,1100.0], maxiters=700)
t_dense, y_dense = GPE.Fitting.predict_model(get_model("logistic_growth"), x, reg_fit.params, 0.0, y[1]; n_curve=150)
push!(produced_paths, save_table("registry_fit_prediction_logistic.csv", DataFrame(time=t_dense, prediction=y_dense)))
push!(fit_rows, (capability="fit_model/predict_model", ok=true, bic=reg_fit.bic, ssr=reg_fit.ssr, output="tables/registry_fit_prediction_logistic.csv"))
hill_fit = fit_model(get_model("theta_logistic_hill_kill"), tx, ty, 2.5; p0=[0.5,1200.0,1.0,0.12,2.0,1.2], maxiters=700)
push!(fit_rows, (capability="fit_model treated Hill kill", ok=true, bic=hill_fit.bic, ssr=hill_fit.ssr, output=""))
cond_results = fit_condition(normalized, "all", [get_model("logistic_growth"),get_model("gompertz_growth"),get_model("theta_logistic_hill_kill")]; group_cols=[:culture_type,:population_type,:dose,:replicate])
push!(produced_paths, save_table("fit_condition_results.csv", cond_results)); push!(fit_rows, (capability="fit_condition", ok=true, bic=minimum(cond_results.bic), ssr=minimum(cond_results.ssr), output="tables/fit_condition_results.csv"))
model_cmp = compare_models(x, y, "logistic", logistic_ode, [0.35,1000.0], "gompertz", gompertz_ode, [0.25,0.0,1400.0]; solver=Tsit5(), bounds1=low_bounds, bounds2=[(0.01,1.0),(0.0,1.0),(200.0,2500.0)], output_csv=joinpath(TABLEDIR,"compare_models_logistic_vs_gompertz.csv"))
push!(produced_paths, joinpath(TABLEDIR,"compare_models_logistic_vs_gompertz.csv")); push!(fit_rows, (capability="compare_models", ok=true, bic=model_cmp.best_model.bic, ssr=model_cmp.best_model.ssr, output="tables/compare_models_logistic_vs_gompertz.csv"))
compare_datasets(x,y,"naive",logistic_ode,[0.35,1000.0],rx,ry,"resistant",logistic_ode,[0.25,1300.0]; solver=Tsit5(), bounds1=low_bounds, bounds2=low_bounds, output_csv=joinpath(TABLEDIR,"compare_datasets_naive_resistant.csv"))
push!(produced_paths, joinpath(TABLEDIR,"compare_datasets_naive_resistant.csv")); push!(fit_rows, (capability="compare_datasets", ok=true, bic=NaN, ssr=NaN, output="tables/compare_datasets_naive_resistant.csv"))
specdict = Dict("exponential"=>(model=exponential_ode,p0=[0.25],bounds=[(0.01,1.0)]), "logistic"=>(model=logistic_ode,p0=[0.35,1000.0],bounds=low_bounds), "gompertz"=>(model=gompertz_ode,p0=[0.25,0.0,1400.0],bounds=[(0.01,1.0),(0.0,1.0),(200.0,2500.0)]))
dict_fits = compare_models_dict(x, y, specdict; default_solver=Tsit5(), output_csv=joinpath(TABLEDIR,"compare_models_dict.csv"))
push!(produced_paths, joinpath(TABLEDIR,"compare_models_dict.csv")); push!(produced_paths, joinpath(TABLEDIR,"compare_models_dict_predictions.csv"))
push!(fit_rows, (capability="compare_models_dict", ok=true, bic=minimum([v.bic for v in values(dict_fits)]), ssr=minimum([v.ssr for v in values(dict_fits)]), output="tables/compare_models_dict.csv"))
fit_three_datasets(x,y,"naive",rx,ry,"resistant",tx,ty,"treated",[0.35,1200.0]; model=logistic_ode, bounds=low_bounds, solver=Tsit5(), output_csv=joinpath(TABLEDIR,"fit_three_datasets.csv"))
three_vec = fit_three_datasets([x,rx,tx], [y,ry,ty]; p0=[0.35,1200.0], model=logistic_ode, bounds=low_bounds, solver=Tsit5())
push!(produced_paths, joinpath(TABLEDIR,"fit_three_datasets.csv")); push!(fit_rows, (capability="fit_three_datasets both overloads", ok=true, bic=NaN, ssr=three_vec.summary.mean_ssr, output="tables/fit_three_datasets.csv"))
fit_summary = DataFrame(fit_rows); push!(produced_paths, save_table("fitting_api_summary.csv", fit_summary)); display(fit_summary)
push!(tour_status, status_row("fitting", "low-level/single/registry/condition/comparison/three-dataset", true, "tables/fitting_api_summary.csv"))


## Joint Fitting And Statistical Analysis


In [ ]:
function joint_competition!(du,u,p,t); S,R=u; rS,rR,KS,KR,aSR,aRS=p; du[1]=rS*S*(1-(S+aSR*R)/KS); du[2]=rR*R*(1-(R+aRS*S)/KR); end
function joint_cooperation!(du,u,p,t); S,R=u; rS,rR,KS,KR,bSR,bRS=p; du[1]=rS*S*(1-S/KS+bSR*R/max(KR,1e-9)); du[2]=rR*R*(1-R/KR+bRS*S/max(KS,1e-9)); end
src = simreg("lotka_volterra_cooperation", [0.44,900.0,0.08,0.31,760.0,0.05], [65.0,45.0])
Sobs = vec(src.states[1,:]) .* (1 .+ 0.02sin.(times)); Robs = vec(src.states[2,:]) .* (1 .+ 0.02cos.(times))
ds = [(x=times,y=Sobs,state_index=1),(x=times,y=Robs,state_index=2)]
joint_fit = run_joint_fit(joint_cooperation!, ds, [65.0,45.0], [0.35,0.25,850.0,750.0,0.05,0.04]; solver=Tsit5(), bounds=[(0.01,1.0),(0.01,1.0),(200,2000),(200,2000),(0,0.25),(0,0.25)], maxiters=700)
joint_cmp = compare_joint_models_dict(ds, [65.0,45.0], Dict("cooperation"=>(model=joint_cooperation!,p0=[0.35,0.25,850.0,750.0,0.05,0.04],bounds=[(0.01,1.0),(0.01,1.0),(200,2000),(200,2000),(0,0.25),(0,0.25)]), "competition"=>(model=joint_competition!,p0=[0.35,0.25,850.0,750.0,0.2,0.2],bounds=[(0.01,1.0),(0.01,1.0),(200,2000),(200,2000),(0,1.5),(0,1.5)])); output_csv=joinpath(TABLEDIR,"joint_model_comparison.csv"))
push!(produced_paths, joinpath(TABLEDIR,"joint_model_comparison.csv"))
joint_pred = DataFrame(time=joint_fit.save_times, sensitive=[u[1] for u in joint_fit.solution.u], resistant=[u[2] for u in joint_fit.solution.u]); push!(produced_paths, save_table("joint_fit_predictions.csv", joint_pred))
pj = plot(title="Joint fit to sensitive and resistant states", xlabel="Time", ylabel="State count"); scatter!(pj,times,Sobs,label="Sensitive data"); scatter!(pj,times,Robs,label="Resistant data"); plot!(pj,joint_pred.time,joint_pred.sensitive,label="Sensitive fit"); plot!(pj,joint_pred.time,joint_pred.resistant,label="Resistant fit")
push!(figure_paths, save_fig("joint_fit_multistate.png", pj))
analysis_rows = NamedTuple[]
for (nm, thunk) in [
    ("leave_one_out_validation", () -> begin r=leave_one_out_validation(x,y,[0.35,1000.0]; model=logistic_ode, bounds=low_bounds, solver=Tsit5()); CSV.write(save_table("loo_predictions.csv", DataFrame(time=x,actual=Float64.(r.actual),predicted=Float64.(r.predictions))), DataFrame(time=x,actual=Float64.(r.actual),predicted=Float64.(r.predictions))); r.rmse end),
    ("k_fold_cross_validation", () -> begin r=k_fold_cross_validation(x,y,[0.35,1000.0]; k_folds=3, model=logistic_ode, bounds=low_bounds, solver=Tsit5()); push!(produced_paths, save_table("kfold_predictions.csv", DataFrame(actual=Float64.(r.actual),predicted=Float64.(r.predictions)))); push!(produced_paths, save_table("kfold_metrics.csv", DataFrame(r.fold_metrics))); r.overall_rmse end)]
    try; val = thunk(); push!(analysis_rows, (capability=nm, ok=true, metric="rmse", value=val, output="tables")); catch err; push!(analysis_rows, (capability=nm, ok=false, metric="error", value=NaN, output=sprint(showerror,err))); end
end
sens = parameter_sensitivity_analysis(x,y,single_fit; perturbation=0.10, model=logistic_ode, solver=Tsit5())
sens_df = DataFrame(param_index=[m.param_index for m in sens.ranking], param_value=[m.param_value for m in sens.ranking], sensitivity_index=[m.sensitivity_index for m in sens.ranking], max_relative_change=[m.max_rel_change for m in sens.ranking])
push!(produced_paths, save_table("parameter_sensitivity_vector_api.csv", sens_df))
reg_sens = parameter_sensitivity_analysis(mono_naive, "logistic_growth"; fitted_params=reg_fit.params, perturbation=0.10); push!(produced_paths, save_table("parameter_sensitivity_registry_api.csv", reg_sens))
resid = residual_analysis(x,y,single_fit; model=logistic_ode, solver=Tsit5(), outlier_threshold=2.0)
resid_df = DataFrame(time=x, actual=y, predicted=Float64.(resid.predicted_values), residual=Float64.(resid.residuals), standardized=Float64.(resid.standardized_residuals)); push!(produced_paths, save_table("residual_analysis.csv", resid_df))
bic_analysis = enhanced_bic_analysis(x,y; models=[exponential_ode,logistic_ode,gompertz_ode], model_names=["Exponential","Logistic","Gompertz"], p0_values=[[0.25],[0.35,1000.0],[0.25,0.0,1400.0]], solver=Tsit5(), max_time=20.0)
bic_df = DataFrame(model=[r.model_name for r in bic_analysis.results], bic=[r.bic for r in bic_analysis.results], aic=[r.aic for r in bic_analysis.results], aicc=[r.aicc for r in bic_analysis.results], rmse=[r.rmse for r in bic_analysis.results], r_squared=[r.r_squared for r in bic_analysis.results], fit_success=[r.fit_success for r in bic_analysis.results]); push!(produced_paths, save_table("enhanced_bic_analysis.csv", bic_df))
push!(analysis_rows, (capability="sensitivity/residual/enhanced_bic", ok=true, metric="count", value=3.0, output="tables"))
analysis_summary = DataFrame(analysis_rows); push!(produced_paths, save_table("statistical_analysis_summary.csv", analysis_summary)); display(analysis_summary)
push!(tour_status, status_row("joint/statistics", "run_joint_fit/compare_joint_models/LOO/k-fold/sensitivity/residual/BIC", all(analysis_summary.ok), "figures/joint_fit_multistate.png"))


## Workflow And Staged Pipeline


In [ ]:
conditions = build_conditions(normalized; condition_cols=[:culture_type,:population_type,:dose,:replicate])
small_conditions = [c for c in conditions if occursin("culture_type=monoculture", c.name) && occursin("population_type=naive", c.name) && occursin("dose=0.0", c.name)]
workflow_fit = GPE.Workflow.fit(get_model("logistic_growth"), small_conditions; shared_params=[:r,:K], n_starts=2, maxiters=180, weighted=true, top_k=3)
rank_result = rank_models(["logistic_growth","gompertz_growth"], small_conditions; n_starts=2, maxiters=180, top_k=3)
plot_files = plot_topk(rank_result; conditions=small_conditions, top_k=3, output_dir=joinpath(OUTDIR,"wf_topk")); append!(produced_paths, plot_files); append!(figure_paths, filter(p -> endswith(lowercase(p), ".png"), plot_files))
exported = export_results(rank_result; output_dir=joinpath(OUTDIR,"wf_export")); append!(produced_paths, collect(values(exported)))
cfg = PipelineConfig("function-tour", ["logistic_growth","gompertz_growth"], 1, 2, 150, 1e-6, 1e-6, true, 2026, joinpath(OUTDIR,"rp"))
cfg_path = joinpath(OUTDIR,"tour_config.toml"); save_config(cfg_path, cfg); loaded_cfg = load_config(cfg_path); push!(produced_paths, cfg_path)
pipeline_result = run_pipeline(normalized; config=cfg, include_models=["logistic_growth","gompertz_growth"], strict_schema=false)
push!(produced_paths, joinpath(OUTDIR,"rp","best_model_summary.csv"))
staged_result = run_staged_pipeline(normalized; config=PipelineConfig("function-tour-staged", ["logistic_growth","gompertz_growth"], 1, 2, 150, 1e-6, 1e-6, true, 2026, joinpath(OUTDIR,"sp")), stages=default_stages(), export_stage_results=false)
manifest_path = joinpath(OUTDIR, "sp", "run_manifest.toml")
save_run_manifest(manifest_path; config=staged_result.config, stage_results=staged_result.stages, parameter_bank=staged_result.parameter_bank, uncertainty_bank=staged_result.uncertainty_bank, failures=staged_result.failures, completed=staged_result.completed, halted_stage=staged_result.halted_stage)
loaded_manifest = load_run_manifest(manifest_path); push!(produced_paths, manifest_path)
stage_summary = DataFrame([
    (stage=s.name, status=s.status, n_conditions=s.n_conditions, selected_model=isnothing(s.selected_model) ? "" : String(s.selected_model),
     best_bic=(hasproperty(s, :result) && s.result !== nothing && haskey(s.result, :ranking) && nrow(s.result.ranking) > 0 ? minimum(s.result.ranking.bic) : NaN))
    for s in staged_result.stages
])
push!(produced_paths, save_table("staged_stage_summary.csv", stage_summary))
parameter_bank_df = isempty(staged_result.parameter_bank) ? DataFrame(stage=String[],param=String[],value=Float64[]) : DataFrame([(stage=s, param=String(k), value=v) for (s,ps) in staged_result.parameter_bank for (k,v) in ps])
push!(produced_paths, save_table("staged_parameter_bank.csv", parameter_bank_df))
boot = bootstrap_stage_uncertainty(get_model("logistic_growth"), normalized[(normalized.culture_type .== "monoculture") .& (normalized.population_type .== "naive") .& (normalized.dose .== 0.0), :]; condition_cols=[:replicate], n_bootstrap=2, n_starts=1, maxiters=120)
boot_df = DataFrame([(param=String(k), metric=m, value=v) for (k,vals) in boot for (m,v) in vals]); push!(produced_paths, save_table("bootstrap_stage_uncertainty.csv", boot_df))
workflow_summary = DataFrame(capability=["build_conditions","fit","rank_models","plot_topk","export_results","save/load config","run_pipeline","run_staged_pipeline","save/load manifest","bootstrap_stage_uncertainty"], ok=[length(conditions)>0,true,nrow(rank_result.ranking)>0,length(plot_files)>0,isfile(exported.summary),loaded_cfg.output_dir==cfg.output_dir,true,staged_result.completed,length(loaded_manifest.completed_stages)>=0,nrow(boot_df)>=0])
push!(produced_paths, save_table("workflow_summary.csv", workflow_summary)); display(workflow_summary)
push!(tour_status, status_row("workflow", "conditions/fit/rank/plot/export/pipeline/staged/bootstrap", all(workflow_summary.ok), "tables/workflow_summary.csv"))


## Figure Gallery And Output Manifest


In [ ]:
pc = plot(title="Manufactured multistage data", xlabel="Time", ylabel="Count")
for g in groupby(manufactured, [:culture_type,:population_type,:treatment_amount,:replicate])
    plot!(pc, g.time, g.count, label="$(g.culture_type[1]) $(g.population_type[1]) dose=$(g.treatment_amount[1]) r$(g.replicate[1])", alpha=0.75)
end
push!(figure_paths, save_fig("manufactured_multistage_timeseries.png", pc))
push!(figure_paths, save_fig("manufactured_model_generators.png", bar(story.model_generator, story.n_rows, title="Rows by manufactured generator", xlabel="Generator", ylabel="Rows", xrotation=35, legend=false)))
pf = plot(title="Logistic registry fit overlay", xlabel="Time", ylabel="Count"); scatter!(pf,x,y,label="Observed"); plot!(pf,t_dense,y_dense,label="Fit"); push!(figure_paths, save_fig("registry_logistic_fit_overlay.png", pf))
push!(figure_paths, save_fig("fitting_api_ssr_summary.png", bar(fit_summary.capability, fit_summary.ssr, title="Fitting API SSR summary", xlabel="Capability", ylabel="SSR", legend=false, xrotation=35)))
cmdf = CSV.read(joinpath(TABLEDIR,"compare_models_dict.csv"), DataFrame); push!(figure_paths, save_fig("compare_models_dict_bic.png", bar(cmdf.Model, cmdf.BIC, title="compare_models_dict BIC", legend=false)))
pr = plot(layout=(1,2), size=(1200,500)); scatter!(pr[1], resid_df.predicted, resid_df.residual, xlabel="Predicted", ylabel="Residual", label="Residuals"); hline!(pr[1],[0.0],label="zero"); qq=sort(resid_df.standardized); qt=[quantile(Normal(),(i-0.5)/length(qq)) for i in eachindex(qq)]; scatter!(pr[2],qt,qq,xlabel="Normal quantile",ylabel="Std residual",label="QQ"); plot!(pr[2],qt,qt,label="reference"); push!(figure_paths, save_fig("residual_diagnostics.png", pr))
push!(figure_paths, save_fig("parameter_sensitivity.png", bar(sens_df.param_index, sens_df.sensitivity_index, title="Parameter sensitivity", xlabel="Parameter", ylabel="Sensitivity index", legend=false)))
push!(figure_paths, save_fig("enhanced_bic_analysis.png", bar(bic_df.model, bic_df.bic, title="Enhanced BIC analysis", xlabel="Model", ylabel="BIC", legend=false)))
if isfile(joinpath(TABLEDIR,"loo_predictions.csv")); loo_df=CSV.read(joinpath(TABLEDIR,"loo_predictions.csv"),DataFrame); p=scatter(loo_df.actual,loo_df.predicted,title="Leave-one-out predictions",xlabel="Actual",ylabel="Predicted",label="LOO"); lim=collect(extrema(loo_df.actual)); plot!(p,lim,lim,label="1:1"); push!(figure_paths, save_fig("loo_predictions.png", p)); end
if isfile(joinpath(TABLEDIR,"kfold_predictions.csv")); cv_df=CSV.read(joinpath(TABLEDIR,"kfold_predictions.csv"),DataFrame); p=scatter(cv_df.actual,cv_df.predicted,title="K-fold predictions",xlabel="Actual",ylabel="Predicted",label="K-fold"); lim=collect(extrema(cv_df.actual)); plot!(p,lim,lim,label="1:1"); push!(figure_paths, save_fig("kfold_predictions.png", p)); end
if nrow(stage_summary)>0; push!(figure_paths, save_fig("staged_pipeline_best_bic.png", bar(stage_summary.stage, stage_summary.best_bic, title="Staged pipeline best BIC", xlabel="Stage", ylabel="BIC", legend=false, xrotation=35))); end
if nrow(parameter_bank_df)>0; push!(figure_paths, save_fig("staged_parameter_bank.png", bar(parameter_bank_df.param, parameter_bank_df.value, title="Staged parameter bank", xlabel="Parameter", ylabel="Value", xrotation=35, legend=false))); end
if nrow(boot_df)>0; mb=unstack(boot_df,:metric,:value); "mean" in names(mb) && push!(figure_paths, save_fig("bootstrap_parameter_means.png", bar(mb.param, mb.mean, title="Bootstrap parameter means", xlabel="Parameter", ylabel="Mean", legend=false))); end
push!(figure_paths, save_fig("registry_model_complexity.png", scatter(reg.n_params, reg.n_states, group=reg.family, title="Registered model complexity", xlabel="Parameters", ylabel="States")))
figure_manifest = DataFrame(path=sort(unique(relpath.(figure_paths, OUTDIR)))); CSV.write(joinpath(OUTDIR,"figure_manifest.csv"), figure_manifest); push!(produced_paths, joinpath(OUTDIR,"figure_manifest.csv"))
push!(tour_status, status_row("figures", "visual gallery for data/fits/statistics/workflows", true, "figure_manifest.csv"))
status_df = DataFrame(tour_status); CSV.write(joinpath(OUTDIR,"capability_status.csv"), status_df); push!(produced_paths, joinpath(OUTDIR,"capability_status.csv"))
all_paths = sort(unique(filter(isfile, produced_paths)))
manifest = DataFrame(path=relpath.(all_paths, OUTDIR), bytes=[filesize(p) for p in all_paths], modified=string.(unix2datetime.(mtime.(all_paths))))
CSV.write(joinpath(OUTDIR,"output_manifest.csv"), manifest)
display(status_df); display(figure_manifest); display(manifest)
println("Generated files: ", nrow(manifest)); println("Generated figures: ", nrow(figure_manifest))
